# Bài 4.4 · LoRA so với fine-tune toàn bộ

**Câu hỏi của buổi này:** chỉ học thêm vài trăm nghìn tham số (LoRA) có theo kịp việc học lại cả 125 triệu tham số không, và đổi lại được gì về thời gian, bộ nhớ, dung lượng?

**SST-2**: câu nhận xét phim tiếng Anh, nhãn tích cực hoặc tiêu cực. Model **RoBERTa-base**; học trên 8 000 câu, chấm trên 872 câu validation.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng hai tập, câu mẫu |
| 2–4. Ba cách học | chỉ head, LoRA r = 8, fine-tune toàn bộ: loss theo bước |
| 5. So ba cách | bảng và biểu đồ: tham số, accuracy, thời gian, bộ nhớ GPU |
| 6. Hạng r | accuracy và số tham số khi r = 1, 4, 8, 32 |
| 7. Merge và tệp lưu | logits trước và sau merge, dung lượng adapter |
| 8. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 12 phút. Bài giảng: [ai-canuc80k · bài 4.4](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/4.4).

In [ ]:
import importlib.util, importlib.metadata, subprocess, sys
# Ảnh Kaggle có sẵn torchao 0.10, quá cũ so với peft mới (peft báo lỗi khi gặp nó); gỡ đi vì notebook không dùng.
if importlib.util.find_spec('torchao') is not None and importlib.metadata.version('torchao') < '0.16':
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True, capture_output=True)
if importlib.util.find_spec('peft') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft'], check=True, capture_output=True)

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import os, time, copy
import torch, transformers, datasets, peft
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
transformers.logging.set_verbosity_error(); datasets.disable_progress_bars()

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL, N_TRAIN, EPOCHS, BATCH, MAXLEN = 'roberta-base', 8000, 2, 32, 64
table(pd.DataFrame([('torch', torch.__version__), ('transformers', transformers.__version__), ('peft', peft.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Tokenize một lần, cắt và đệm về 64 token; câu SST-2 ngắn nên 64 token giữ gần như trọn câu.

In [ ]:
with quiet():
    ds = datasets.load_dataset('stanfordnlp/sst2')
    tok = AutoTokenizer.from_pretrained(MODEL)
train = ds['train'].shuffle(seed=0).select(range(N_TRAIN))
val = ds['validation']

def encode(split):
    e = tok(list(split['sentence']), truncation=True, max_length=MAXLEN, padding='max_length', return_tensors='pt')
    return list(zip(e['input_ids'], e['attention_mask'], torch.tensor(list(split['label']))))
train_data, val_data = encode(train), encode(val)
table(pd.DataFrame([('train', len(train_data), np.mean(train['label'])), ('validation', len(val_data), np.mean(val['label']))],
                   columns=['tập', 'số câu', 'tỉ lệ tích cực']), 'SST-2', fmt={'tỉ lệ tích cực': '{:.3f}'})
table(pd.DataFrame([(train[i]['sentence'], 'tích cực' if train[i]['label'] else 'tiêu cực') for i in range(5)], columns=['câu', 'nhãn']), 'Năm câu train đầu')

## 2. Một hàm huấn luyện dùng chung

Các cách chỉ khác nhau ở **tham số nào được học**; vòng lặp, lịch learning rate và số epoch giữ nguyên, trừ learning rate: fine-tune toàn bộ dùng learning rate nhỏ để không phá trọng số đã học. Hàm ghi loss từng bước, accuracy, thời gian và bộ nhớ GPU đỉnh.

In [ ]:
def collate(batch):
    ids, mask, y = zip(*batch)
    return torch.stack(ids), torch.stack(mask), torch.stack(y)
train_dl = DataLoader(train_data, BATCH, shuffle=True, collate_fn=collate, generator=torch.Generator().manual_seed(0))
val_dl = DataLoader(val_data, 128, collate_fn=collate)

def evaluate(model):
    model.eval(); hit = 0
    with torch.inference_mode(), torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
        for ids, mask, y in val_dl:
            hit += (model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits.argmax(1).cpu() == y).sum().item()
    return hit / len(val_data)

def fit(model, lr):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=EPOCHS * len(train_dl), pct_start=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')
    if device == 'cuda': torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t, losses = time.time(), []
    for ep in range(EPOCHS):
        model.train()
        for ids, mask, y in train_dl:
            with torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
                loss = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=y.to(device)).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            losses.append(loss.item())
    sec = time.time() - t
    mem = torch.cuda.max_memory_allocated() / 2**30 if device == 'cuda' else float('nan')
    return evaluate(model), sec, mem, losses

trainable = lambda model: sum(p.numel() for p in model.parameters() if p.requires_grad)
def new_model():
    with quiet():
        return AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(device)
results, curves = [], {}
N_ALL = sum(p.numel() for p in new_model().parameters())

## 3. Chỉ học head

Head phân loại của RoBERTa gồm một lớp dense và một lớp ra 2 logit; mọi tham số khác bị đóng băng.

In [ ]:
m = new_model()
for n, p in m.named_parameters(): p.requires_grad = n.startswith('classifier.')
acc, sec, mem, curves['chỉ head'] = fit(m, 1e-3)
results.append(('chỉ head', trainable(m), acc, sec, mem)); del m
check(f'chỉ head học {results[-1][1]:,} tham số', results[-1][1] < 1e6)

## 4. LoRA r = 8, rồi fine-tune toàn bộ

Mỗi ma trận $W$ ($768 \times 768$) của query và value có thêm $B$ ($768 \times r$) và $A$ ($r \times 768$); đầu ra là $Wx + \frac{\alpha}{r} BAx$. $B$ khởi tạo bằng 0, nên lúc đầu model y hệt bản gốc. Head phân loại vẫn học đầy đủ (`modules_to_save`).

In [ ]:
def make_lora(r):
    cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.1, target_modules=['query', 'value'], modules_to_save=['classifier'], task_type='SEQ_CLS')
    return get_peft_model(new_model(), cfg)

lora = make_lora(8)
lora_only = sum(p.numel() for n, p in lora.named_parameters() if 'lora_' in n)
check(f'ma trận LoRA: {lora_only:,} = 12 tầng × 2 ma trận × (768·8 + 8·768)', lora_only == 12 * 2 * (768 * 8 + 8 * 768))
acc, sec, mem, curves['LoRA r=8'] = fit(lora, 1e-3)
results.append(('LoRA r=8', trainable(lora), acc, sec, mem))

m = new_model()
acc, sec, mem, curves['fine-tune toàn bộ'] = fit(m, 2e-5)
results.append(('fine-tune toàn bộ', trainable(m), acc, sec, mem))
full_bytes = sum(p.numel() * p.element_size() for p in m.parameters())
del m
if device == 'cuda': torch.cuda.empty_cache()

fig, ax = plt.subplots(figsize=(10, 3.6))
for (n, l), c in zip(curves.items(), [C['gray'], C['blue'], C['clay']]):
    ax.plot(pd.Series(l).rolling(25, min_periods=1).mean(), color=c, lw=2, label=n)
ax.axvline(len(train_dl), color='#999', ls=':', lw=1)
ax.set(xlabel='bước (đường chấm: hết epoch 1)', ylabel='loss (trung bình trượt 25 bước)', title='Loss train của ba cách học'); ax.legend()
plt.tight_layout(); plt.show()

## 5. So ba cách

In [ ]:
res = pd.DataFrame(results, columns=['cách', 'tham số được học', 'val acc', 'giây', 'GPU đỉnh (GB)'])
res['% tham số'] = res['tham số được học'] / N_ALL
table(res, f'RoBERTa-base ({N_ALL:,} tham số), {EPOCHS} epoch, {N_TRAIN} câu', fmt={'tham số được học': '{:,}', 'val acc': '{:.3f}', 'giây': '{:.0f}', 'GPU đỉnh (GB)': '{:.2f}', '% tham số': '{:.2%}'}, best='val acc')

fig, axes = plt.subplots(1, 4, figsize=(17, 3.4))
cols = [C['gray'], C['blue'], C['clay']]
for ax, (col, t, f) in zip(axes, [('tham số được học', 'tham số được học (thang log)', '{:,.0f}'), ('val acc', 'accuracy validation', '{:.3f}'),
                                  ('giây', 'thời gian train (giây)', '{:.0f}'), ('GPU đỉnh (GB)', 'bộ nhớ GPU đỉnh (GB)', '{:.2f}')]):
    ax.bar(res['cách'], res[col], color=cols); ax.set_title(t, fontsize=11); ax.tick_params(axis='x', labelsize=9); bar_labels(ax, f)
    if col == 'tham số được học': ax.set_yscale('log')
    if col == 'val acc': ax.set_ylim(0.5, 1.0)
plt.tight_layout(); plt.show()
h, lo, fu = res.iloc[0], res.iloc[1], res.iloc[2]
check('LoRA vượt chỉ học head', lo['val acc'] > h['val acc'])
note(f"LoRA học <b>{lo['% tham số']:.2%}</b> số tham số mà đạt {lo['val acc']:.3f}, so với {fu['val acc']:.3f} của fine-tune toàn bộ. "
     f"Nó vẫn phải lan truyền ngược qua cả mạng, nên tốn {lo['giây'] / fu['giây']:.0%} thời gian và {lo['GPU đỉnh (GB)'] / fu['GPU đỉnh (GB)']:.0%} bộ nhớ của fine-tune toàn bộ, nhiều hơn hẳn chỉ học head.")

## 6. Hạng r

$r$ càng lớn thì $BA$ càng có thể biểu diễn nhiều kiểu thay đổi, nhưng số tham số tăng tuyến tính theo $r$. Thử $r$ = 1, 4, 32 (đã có $r$ = 8).

In [ ]:
rank_rows = [(8, lo['tham số được học'], lo['val acc'], lo['giây'])]
for r in [1, 4, 32]:
    torch.manual_seed(0)
    mr = make_lora(r); acc, sec, _, _ = fit(mr, 1e-3)
    rank_rows.append((r, trainable(mr), acc, sec)); del mr
rank = pd.DataFrame(sorted(rank_rows), columns=['r', 'tham số được học', 'val acc', 'giây'])
table(rank, 'LoRA theo hạng r', fmt={'tham số được học': '{:,}', 'val acc': '{:.3f}', 'giây': '{:.0f}'}, best='val acc')
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(rank['r'], rank['val acc'], marker='o', color=C['blue'], lw=2)
for _, row in rank.iterrows(): ax.annotate(f"{row['tham số được học'] / 1e3:.0f}k tham số", (row['r'], row['val acc']), xytext=(4, -14), textcoords='offset points', fontsize=9)
ax.axhline(fu['val acc'], color=C['clay'], ls='--'); ax.text(1, fu['val acc'] + 0.002, 'fine-tune toàn bộ', color=C['clay'], fontsize=9)
ax.set_xscale('log', base=2); ax.set(xlabel='hạng r', ylabel='accuracy validation', title='Tăng r: số tham số tăng đều, accuracy gần như không đổi')
plt.tight_layout(); plt.show()
note(f"Từ r = 1 đến r = 32, số tham số LoRA gấp 32 lần mà accuracy chỉ từ {rank['val acc'].iloc[0]:.3f} đến {rank['val acc'].iloc[-1]:.3f}: "
     f"thay đổi cần cho bài này nằm trong một không gian rất ít chiều.")

## 7. Merge và tệp lưu

Sau khi học xong, cộng thẳng $\frac{\alpha}{r} BA$ vào $W$. Model merge không còn nhánh LoRA, chạy nhanh như model gốc, và phải cho **cùng logits** với model chưa merge. Adapter chỉ chứa $A$, $B$ và head.

In [ ]:
lora.eval()
ids, mask, _ = next(iter(val_dl)); ids, mask = ids[:16].to(device), mask[:16].to(device)
with torch.inference_mode():
    before = lora(input_ids=ids, attention_mask=mask).logits.float()
merged = copy.deepcopy(lora).merge_and_unload().eval()
with torch.inference_mode():
    after = merged(input_ids=ids, attention_mask=mask).logits.float()
gap = (before - after).abs().max().item()
check(f'logits trước và sau merge chênh lớn nhất {gap:.1e}', gap < 1e-3)
check('model merge không còn lớp LoRA', not any('lora_' in n for n, _ in merged.named_parameters()))

lora.save_pretrained('/tmp/lora_adapter')
adapter_mb = sum(os.path.getsize(os.path.join('/tmp/lora_adapter', f)) for f in os.listdir('/tmp/lora_adapter')) / 2**20
size = pd.DataFrame([('adapter LoRA r=8', adapter_mb), ('model fine-tune toàn bộ (float32)', full_bytes / 2**20)], columns=['tệp', 'MB'])
table(size, 'Dung lượng cần lưu cho mỗi bài', fmt={'MB': '{:.1f}'})
note(f'Adapter <b>{adapter_mb:.1f} MB</b> so với {full_bytes / 2**20:.0f} MB: một model gốc dùng chung được cho nhiều bài, mỗi bài một adapter nhỏ.')

mine = ['a moving and beautifully acted film', 'the plot is thin and the jokes fall flat', 'not bad at all', 'i expected more from this director']
e = tok(mine, padding=True, return_tensors='pt').to(device)
with torch.inference_mode():
    p = merged(**e).logits.float().softmax(-1)[:, 1].cpu().numpy()
table(pd.DataFrame({'câu tự viết': mine, 'xác suất tích cực': p}), 'Model LoRA đã merge đọc câu mới', fmt={'xác suất tích cực': '{:.3f}'})

## 8. Tóm tắt

In [ ]:
summary = [('val acc: chỉ head / LoRA r=8 / toàn bộ', f"{h['val acc']:.3f} / {lo['val acc']:.3f} / {fu['val acc']:.3f}"),
           ('tham số được học: LoRA / toàn bộ', f"{lo['tham số được học']:,} / {fu['tham số được học']:,}"),
           ('thời gian, bộ nhớ của LoRA so với toàn bộ', f"{lo['giây'] / fu['giây']:.0%}, {lo['GPU đỉnh (GB)'] / fu['GPU đỉnh (GB)']:.0%}"),
           ('val acc theo r = 1, 4, 8, 32', ' / '.join(f'{v:.3f}' for v in rank['val acc'])),
           ('chênh logits sau merge', f'{gap:.1e}'),
           ('adapter / model đầy đủ', f'{adapter_mb:.1f} MB / {full_bytes / 2**20:.0f} MB')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** thêm `'key'` và `'dense'` vào `target_modules`; giảm `N_TRAIN` xuống 1 000 để xem cách nào chịu thiếu dữ liệu tốt hơn; tăng `EPOCHS` cho chỉ học head.